# Computing gradients of propagation by Automatic differentiation

In this example, we demonstrate how gradient of propagation module can be computed using automatic differentation. For a minimal working example, here we unpack the `ScipyExpm` method and compute the gradients by AD. Finally, we compare the computed gradients with GOAT for verification.

Since we use a modular architecture for the package, the propagation method computes the equation of motion (EOM) for some given times (specified by the initial and final time, and the resolution). Thus the gradient in this case has to be computed using the chain rule

$$\frac{\partial \ket{\psi(t)}}{\partial \alpha} = \sum_{\tau \in [0, t]} \frac{\partial \ket{\psi(t)}}{\partial H(\tau)} \frac{\partial H(\tau)}{\partial \alpha} $$

where $\alpha$ is some pulse parameter. The first term on the right can be computed by automatic differentiation of the propagation method, and the second term is obtained through the derivative of the Hamiltonian (or the EOM).


Let's define a simple system of a driven qubit. 

In [ ]:
import numpy as np

from paraqeet.eom.schroedinger_equation import SchroedingerEquation
from paraqeet.hamiltonian.drive import Drive
from paraqeet.hamiltonian.qubit import QubitHamiltonian
from paraqeet.optimization_map import OptimizationMap
from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import ConstantEnvelope
from paraqeet.signal.iq_mixer import IQMixer

In [ ]:
freq_q = 4.8e9
omega_q = 2 * np.pi * freq_q

envelope = ConstantEnvelope()
gen = IQMixer(envelopes=[envelope])

qubit_hamiltonian = QubitHamiltonian(frequency=Quantity(omega_q, 0.8 * omega_q, 1.2 * omega_q, unit="Hz"), drives=[])
drive = Drive(qubit_hamiltonian.sigma_x, gen)
qubit_hamiltonian.drives = [drive]

schrgl = SchroedingerEquation(
    hamiltonian_func=qubit_hamiltonian.get_value,
    hamiltonian_gradient_func=qubit_hamiltonian.get_gradient,
)

params_tone = envelope.get_parameters()
params_gen = gen.get_parameters()
params_gen

Add parameters to the optmap and register the optimizable parameters

In [ ]:
t_simu = 10e-9
params_gen[0].set_value(0.8 * np.pi / t_simu)
params_gen[2].set_value(1.01 * omega_q)

optmap = OptimizationMap()
optmap.add(gen, [params_gen[0], params_gen[1], params_gen[2]])
optmap.register_params_with_optimizables()

Here we unpack the inner workings of `ScipyExpm` to construct an easy example to work with. Lets first compute the EOM for the interpolated times (given a resolution), which we will then pass to the propagate method

In [ ]:
import jax.numpy as jnp

from paraqeet.propagation.utils import construct_times

init = np.array([[1.0], [0.0j]])  # |0>
target = np.array([[0.0j], [1.0]])  # |1>

time = jnp.array([0.0, t_simu])
resolution = 100e9
eom_func = schrgl.get_value

time_grid = []
eoms = []

for ti in range(1, len(time)):
    times, dt = construct_times(time, ti, resolution)
    eom = eom_func(times + dt / 2) * dt
    time_grid.append(times + dt / 2)
    eoms.append(eom)

time_grid = jnp.hstack(time_grid)
eoms = jnp.hstack(eoms)
steps_arr = jnp.arange(0, len(time_grid), 1)

The following are the main functions in `ScipyExpm`

In [ ]:
from jax import jit
from jax.lax import scan
from jax.scipy.linalg import expm


@jit
def propagate_in_time(psis_t, eom, steps_arr):
    """Propagate the system in time.

    Iteratively propagate state/states (psis_t) according
    to the equation of motion (eom). The eom is exponentiated using
    `jax.scipy.linalg.expm` to compute the propagators.
    The iterations use `jax.lax.scan` to avoid compilation overhead.

    Parameters
    ----------
    psis_t: Array
        State/states at time 't'.
    eom: Array
        Equation of motion for a list of times.
    steps_arr: Array
        Array from 0 to the length of the List of time, in steps of 1
        representing the iteration index.

    Returns
    -------
    Array
        Returns the evolved state.

    """

    def propagate_body(psis_t, index):
        psis_t = propagate_psi(eom[index], psis_t)
        return psis_t, psis_t

    psis_t, _ = scan(propagate_body, psis_t, steps_arr)

    return psis_t


@staticmethod
@jit
def propagate_psi(eom_matrix, psis_t):
    """Propagate the state/states (psis_t).

    Parameters
    ----------
    eom_matrix : Array
        The equations of motion matrix.
    psis_t : Array
        State/states at time 't'.

    Returns
    -------
    Array
        Returns the evolved state.

    """
    return expm(eom_matrix) @ psis_t

In [ ]:
propagate_in_time(jnp.array(init), eoms, steps_arr)

Let's compute the gradient of the evolved state with respect to the Hamiltonian. Here we use the `get_value_and_jacobian_rev` function defined in autograd_utils

In [ ]:
from paraqeet.autograd_utils import get_value_and_jacobian_rev

prop_grad_func = get_value_and_jacobian_rev(propagate_in_time, argnums=1)
val, grads = prop_grad_func(jnp.array(init), eoms, steps_arr)

In [ ]:
grads.shape

We need to sum over the gradient of the EOM at different time

In [ ]:
_, schrgl_grads = schrgl.get_value_and_gradient(time_grid)

In [ ]:
schrgl_grads.shape

Here the second index is for the number of parameters.

In [ ]:
grads1 = jnp.transpose(grads, axes=(2, 0, 1, 3, 4))
grads1.shape

We need to sum over times and the last two indices. Using the notation "t -> times", "n,m-> dim of state vector", "p -> no. of parameters", and "j, k -> dim of Hamiltonian", we perform the following summation

In [ ]:
final_grads = jnp.einsum("tnmjk, tpjk -> pnm", grads1, schrgl_grads) * dt
final_grads.shape

## Comparison with GOAT

In [ ]:
freq_q = 4.8e9
omega_q = 2 * np.pi * freq_q

envelope = ConstantEnvelope()
gen = IQMixer(envelopes=[envelope])

qubit_hamiltonian = QubitHamiltonian(frequency=Quantity(omega_q, 0.8 * omega_q, 1.2 * omega_q, unit="Hz"), drives=[])
drive = Drive(qubit_hamiltonian.sigma_x, gen)
qubit_hamiltonian.drives = [drive]

schrgl = SchroedingerEquation(
    hamiltonian_func=qubit_hamiltonian.get_value,
    hamiltonian_gradient_func=qubit_hamiltonian.get_gradient,
)

params_tone = envelope.get_parameters()
params_gen = gen.get_parameters()
params_gen

In [ ]:
t_simu = 10e-9
params_gen[0].set_value(0.8 * np.pi / t_simu)
params_gen[2].set_value(1.01 * omega_q)

In [ ]:
from paraqeet.propagation.expm_goat import ExpmGOAT

init = np.array([[1.0], [0.0]])  # |0>
target = np.array([[0.0], [1.0]])  # |1>

times = np.array([0.0, t_simu])

prop = ExpmGOAT(
    eom_func=schrgl.get_value, eom_gradient_func=schrgl.get_gradient, resolution=100e9, initial_state=init
)  # implicit timestep is 1 / resolution

In [ ]:
optmap = OptimizationMap()
optmap.add(gen, [params_gen[0], params_gen[1], params_gen[2]])
optmap.register_params_with_optimizables()

In [ ]:
val, grads = prop.get_value_and_gradient(np.array([0.0, t_simu]))

In [ ]:
jnp.allclose(grads[-1], final_grads)

We can see that the gradients from AD match with the gradient computed using GOAT.